In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from lightgbm import LGBMClassifier

In [20]:
TRAIN = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/train.csv")
TEST = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/test.csv")
SUBMIT = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv")

In [21]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

def preprocess_data(df, target_col="satisfaction", drop_nan_col="Arrival Delay in Minutes", preprocessor=None):
    df_clean = df.copy()

    if "Arrival Delay in Minutes" in df_clean.columns and "Departure Delay in Minutes" in df_clean.columns:
        df_clean["Inflight Delay Change"] = df_clean["Arrival Delay in Minutes"] - df_clean["Departure Delay in Minutes"]
        df_clean = df_clean.drop(columns=["Arrival Delay in Minutes", "Departure Delay in Minutes"])

    Y = df_clean[target_col].astype(int) if target_col in df_clean.columns else None
    X = df_clean.drop(columns=[target_col, "id"], errors="ignore")
    nom_cat_col = ["Gender", "Customer Type", "Type of Travel"]

    if preprocessor is None:
        nom_map = [X[col].dropna().unique().tolist() for col in nom_cat_col]
        preprocessor = ColumnTransformer([
            ("nom", OrdinalEncoder(categories=nom_map, handle_unknown="use_encoded_value", unknown_value=-1), nom_cat_col),
            ("class", OrdinalEncoder(categories=[["Eco", "Eco Plus", "Business"]], handle_unknown="use_encoded_value", unknown_value=-1), ["Class"])
        ], remainder="passthrough", verbose_feature_names_out=False)
        X_processed = preprocessor.fit_transform(X)
    else:
        X_processed = preprocessor.transform(X)

    X_df = pd.DataFrame(X_processed, columns=preprocessor.get_feature_names_out(), index=X.index)[X.columns]

    num1_col = ["Age", "Flight Distance", "Inflight Delay Change"]
    X_df[num1_col] = X_df[num1_col].astype("float64")
    
    cat1_col = [col for col in X_df.columns if col not in num1_col]
    X_df[cat1_col] = X_df[cat1_col].astype("int64").astype("category")
    return X_df, Y, preprocessor

In [22]:
X_train, y_train, fitted_preprocessor = preprocess_data(TRAIN, target_col="satisfaction")

X_test, _, _ = preprocess_data(TEST, target_col="satisfaction", preprocessor=fitted_preprocessor)

In [23]:
model = LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=90, n_jobs=-1, verbosity=-1)
model.fit(X_train, y_train)

test_preds = model.predict_proba(X_test)[:, 1]

In [25]:
SUBMIT['satisfaction'] = test_preds

# Save to CSV for Kaggle submission
SUBMIT.to_csv("submission.csv", index=False)

# Quick sanity check
print("Shape:", SUBMIT.shape)
print("Missing values:", SUBMIT.isna().sum().sum())
print(SUBMIT.head())

Shape: (299844, 2)
Missing values: 0
       id  satisfaction
0  699635      0.180166
1  699636      0.876931
2  699637      0.894882
3  699638      0.073470
4  699639      0.053267


,id,satisfaction
0,699635,0.443573
1,699636,0.443573
2,699637,0.443573
3,699638,0.443573
4,699639,0.443573
